Análisis de Datos CSV ASOCIACIÓN

iMPORTACIÓN DE LIBRERIAS

In [25]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

CARGAR ARCHIVO CSV

In [26]:
url_asociacion = "https://raw.githubusercontent.com/eduardorivas2517502022/Parcial4-RivasEduardo-2517502022/refs/heads/main/csv/clave_G_asociacion.csv"

df_asociacion = pd.read_csv(url_asociacion)

df_asociacion.head()

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,G-T0001,G-C0072,2026-02-20,Construccion,Arena,1,Tienda
1,G-T0001,G-C0072,2026-02-20,Herramientas,Destornillador,1,Tienda
2,G-T0001,G-C0072,2026-02-20,Pintura,Lija,3,Tienda
3,G-T0002,G-C0057,2026-01-26,Electricidad,Cable,2,App
4,G-T0002,G-C0057,2026-01-26,Electricidad,Interruptor,1,App


REVISAR ESTRUCTURA DEL DATASET

In [27]:
df_asociacion.info()
df_asociacion.isnull().sum()
df_asociacion.duplicated().sum()
df_asociacion.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 622 entries, 0 to 621
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   transaccion_id  622 non-null    object
 1   cliente_id      622 non-null    object
 2   fecha           622 non-null    object
 3   categoria       622 non-null    object
 4   item            622 non-null    object
 5   cantidad        622 non-null    int64 
 6   canal           621 non-null    object
dtypes: int64(1), object(6)
memory usage: 34.1+ KB


Index(['transaccion_id', 'cliente_id', 'fecha', 'categoria', 'item',
       'cantidad', 'canal'],
      dtype='object')

CREACIÓN DE TRANSACCIONES Y Convertir a formato transaccional

In [22]:
transacciones = df_asociacion.groupby('transaccion_id')['item'].apply(list).tolist()

te = TransactionEncoder()
te_array = te.fit(transacciones).transform(transacciones)

df_transacciones = pd.DataFrame(te_array, columns=te.columns_)

df_transacciones.head()

,Alicate,Arena,Brocha,Cable,Cemento,Clavos,Codo,Destornillador,Foco,Interruptor,Lija,Llave_paso,Martillo,Pegamento_PVC,Pintura_blanca,Rodillo,Taladro,Tomacorriente,Tornillos,Tubo_PVC
0,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False
1,False,False,False,True,False,False,False,False,False,True,False,True,True,False,False,False,False,True,False,False
2,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,False,False
4,False,True,False,True,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True


Aplicar algoritmo Apriori

In [23]:
frequent_itemsets = apriori(
    df_transacciones,
    min_support=0.05,
    use_colnames=True
)

frequent_itemsets.head()

,support,itemsets
0,0.097436,(Alicate)
1,0.102564,(Arena)
2,0.246154,(Brocha)
3,0.225641,(Cable)
4,0.123077,(Cemento)


Generar reglas de asociación

In [28]:
reglas = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.4
)

Ordenar reglas por Lift

In [29]:
reglas = reglas.sort_values(by='lift', ascending=False)

reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

,antecedents,consequents,support,confidence,lift
13,"(Brocha, Rodillo)",(Pintura_blanca),0.092308,0.900000,3.734043
11,"(Pintura_blanca, Brocha)",(Rodillo),0.092308,0.818182,3.468379
12,"(Pintura_blanca, Rodillo)",(Brocha),0.092308,0.750000,3.046875
6,(Tubo_PVC),(Pegamento_PVC),0.128205,0.625000,2.769886
7,(Pegamento_PVC),(Tubo_PVC),0.128205,0.568182,2.769886
4,(Tomacorriente),(Cable),0.112821,0.523810,2.321429
5,(Cable),(Tomacorriente),0.112821,0.500000,2.321429
8,(Pintura_blanca),(Rodillo),0.123077,0.510638,2.164662
9,(Rodillo),(Pintura_blanca),0.123077,0.521739,2.164662
10,(Taladro),(Tornillos),0.087179,0.500000,2.031250


Mostrar las 10 reglas más importantes

In [30]:
top_10 = reglas[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
].head(10)

top_10

,antecedents,consequents,support,confidence,lift
13,"(Brocha, Rodillo)",(Pintura_blanca),0.092308,0.900000,3.734043
11,"(Pintura_blanca, Brocha)",(Rodillo),0.092308,0.818182,3.468379
12,"(Pintura_blanca, Rodillo)",(Brocha),0.092308,0.750000,3.046875
6,(Tubo_PVC),(Pegamento_PVC),0.128205,0.625000,2.769886
7,(Pegamento_PVC),(Tubo_PVC),0.128205,0.568182,2.769886
4,(Tomacorriente),(Cable),0.112821,0.523810,2.321429
5,(Cable),(Tomacorriente),0.112821,0.500000,2.321429
8,(Pintura_blanca),(Rodillo),0.123077,0.510638,2.164662
9,(Rodillo),(Pintura_blanca),0.123077,0.521739,2.164662
10,(Taladro),(Tornillos),0.087179,0.500000,2.031250
